In [1]:
!pip install pandas requests geopandas shapely pyproj

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 4.4 MB/s eta 0:00:01
   ----------------------------- ---------- 1.3/1.8 MB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 3.2 MB/s  0:00:00
   ---------------------------------------- 0.0/6.7 MB ? eta -:--:--
   --- ------------------------------------ 0.5/6.7 MB 4.5 MB/s eta 0:00:02
   --------- ------------------------------ 1.6/6.7 MB 3.7 MB/s eta 0:00:02
   ------------ --------------------------- 2.1/6.7 MB 3.9 MB/s eta 0:00:02
   --------------- ------------------------ 2.6/6.7 MB 3.2 MB/s eta 0:00:02
   -------------------- ------------------- 3.4/6.7 MB 3.3 MB/s eta 0:00:01
   ----------------------- ---------------- 3.9/6.7 MB 3.3 MB/s eta 0:00:01
   ---------------------------- ----------- 4.7/6.7 MB 3.3 MB/s eta 0:00:01
   ------------------------------- -------- 5.2/6.7 MB 3.1 MB/s eta 0:00:01
   --------------------------

# Define your nine locations

In [2]:

import pandas as pd
import requests
import time

ZONES = [
    {"zone_id": 0, "zone_name": "Bandra",
     "latitude": 19.0600, "longitude": 72.8500},

    {"zone_id": 1, "zone_name": "Santacruz",
     "latitude": 19.0800, "longitude": 72.8400},

    {"zone_id": 2, "zone_name": "Vile Parle",
     "latitude": 19.1000, "longitude": 72.8500},

    {"zone_id": 3, "zone_name": "Andheri",
     "latitude": 19.1200, "longitude": 72.8500},

    {"zone_id": 4, "zone_name": "Jogeshwari",
     "latitude": 19.1400, "longitude": 72.8500},

    {"zone_id": 5, "zone_name": "Goregaon",
     "latitude": 19.1600, "longitude": 72.8500},

    {"zone_id": 6, "zone_name": "Malad",
     "latitude": 19.1800, "longitude": 72.8500},

    {"zone_id": 7, "zone_name": "Kandivali",
     "latitude": 19.2000, "longitude": 72.8500},

    {"zone_id": 8, "zone_name": "Borivali",
     "latitude": 19.2300, "longitude": 72.8500}
]

zones_df = pd.DataFrame(ZONES)

print(zones_df)


   zone_id   zone_name  latitude  longitude
0        0      Bandra     19.06      72.85
1        1   Santacruz     19.08      72.84
2        2  Vile Parle     19.10      72.85
3        3     Andheri     19.12      72.85
4        4  Jogeshwari     19.14      72.85
5        5    Goregaon     19.16      72.85
6        6       Malad     19.18      72.85
7        7   Kandivali     19.20      72.85
8        8    Borivali     19.23      72.85


# Download the geographical data

In [3]:

import requests
import json
import time

# Bounding box covering the nine locations
# south, west, north, east
BBOX = (19.00, 72.78, 19.30, 72.95)

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

query = f"""
[out:json][timeout:180];

(
  way["natural"="coastline"]({BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]});

  way["waterway"~"river|stream|canal|drain|ditch|tidal_channel"]
  ({BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]});
);

out geom;
"""

print("Requesting geographical data from OpenStreetMap...")

response = requests.post(
    OVERPASS_URL,
    data={"data": query},
    headers={"User-Agent": "MumbaiFloodRiskStudentProject/1.0"},
    timeout=240
)

response.raise_for_status()

osm_data = response.json()

print("Data extraction completed!")
print("Total features:", len(osm_data["elements"]))

# Save original response for future processing
with open("mumbai_osm_terrain_raw.json", "w") as file:
    json.dump(osm_data, file, indent=2)

print("Raw OSM data saved!")


Requesting geographical data from OpenStreetMap...
Data extraction completed!
Total features: 771
Raw OSM data saved!


# Check the extracted features

In [4]:

elements = osm_data["elements"]

coastline_count = 0
waterway_count = 0

waterway_types = {}

for element in elements:

    tags = element.get("tags", {})

    if tags.get("natural") == "coastline":
        coastline_count += 1

    if "waterway" in tags:
        waterway_count += 1

        waterway_type = tags["waterway"]

        waterway_types[waterway_type] = (
            waterway_types.get(waterway_type, 0) + 1
        )

print("Coastline features:", coastline_count)
print("Waterway features:", waterway_count)

print("\nWaterway categories:")
print(waterway_types)


Coastline features: 22
Waterway features: 749

Waterway categories:
{'drain': 504, 'river': 76, 'stream': 166, 'canal': 3}


# Calculate distances

In [5]:

import geopandas as gpd
from shapely.geometry import LineString, Point

coastline_lines = []
nullah_lines = []

for element in osm_data["elements"]:

    tags = element.get("tags", {})
    geometry = element.get("geometry", [])

    if len(geometry) < 2:
        continue

    coords = [
        (point["lon"], point["lat"])
        for point in geometry
    ]

    line = LineString(coords)

    if tags.get("natural") == "coastline":
        coastline_lines.append(line)

    if tags.get("waterway") in [
        "river", "stream", "canal",
        "drain", "ditch", "tidal_channel"
    ]:
        nullah_lines.append(line)

print("Coastline lines:", len(coastline_lines))
print("Waterway lines:", len(nullah_lines))


Coastline lines: 22
Waterway lines: 749


# Creating geographical layers

In [6]:

coast_gdf = gpd.GeoDataFrame(
    geometry=coastline_lines,
    crs="EPSG:4326"
).to_crs("EPSG:32643")

water_gdf = gpd.GeoDataFrame(
    geometry=nullah_lines,
    crs="EPSG:4326"
).to_crs("EPSG:32643")

print("Geographical layers created!")


Geographical layers created!


# Calculating nearest distances for all nine locations

In [10]:

distance_results = []

for zone in ZONES:

    point = Point(
        zone["longitude"],
        zone["latitude"]
    )

    point_gdf = gpd.GeoSeries(
        [point],
        crs="EPSG:4326"
    ).to_crs("EPSG:32643")

    projected_point = point_gdf.iloc[0]

    # Distance to nearest coastline
    if not coast_gdf.empty:
        coast_distance = coast_gdf.distance(
            projected_point
        ).min()
    else:
        coast_distance = None

    # Distance to nearest mapped waterway
    if not water_gdf.empty:
        nullah_distance = water_gdf.distance(
            projected_point
        ).min()
    else:
        nullah_distance = None

    distance_results.append({
        "zone_id": zone["zone_id"],
        "zone_name": zone["zone_name"],
        "distance_to_coast_m": coast_distance,
        "distance_to_nearest_waterway_m": nullah_distance
})

    print(
        zone["zone_name"],
        "| Coast:", coast_distance,
        "| Waterway:", nullah_distance
    )

terrain_df = pd.DataFrame(distance_results)

terrain_df.to_csv(
    "mumbai_terrain_geographical_features.csv",
    index=False
)

print("\nDistance calculations completed!")
print(terrain_df)


Bandra | Coast: 1777.1052135479708 | Waterway: 669.1858957639839
Santacruz | Coast: 1655.4870948342646 | Waterway: 652.8914859160317
Vile Parle | Coast: 2530.5858950194192 | Waterway: 647.285366995661
Andheri | Coast: 3021.0117428919543 | Waterway: 755.9816326484626
Jogeshwari | Coast: 3876.4003397556885 | Waterway: 136.62696514364978
Goregaon | Coast: 5075.700191882443 | Waterway: 258.2243496082551
Malad | Coast: 5669.582388071132 | Waterway: 1007.0566256565412
Kandivali | Coast: 5614.00559089757 | Waterway: 439.30726443197926
Borivali | Coast: 6693.266464788249 | Waterway: 998.8021632562271

Distance calculations completed!
   zone_id   zone_name  distance_to_coast_m  distance_to_nearest_waterway_m
0        0      Bandra          1777.105214                      669.185896
1        1   Santacruz          1655.487095                      652.891486
2        2  Vile Parle          2530.585895                      647.285367
3        3     Andheri          3021.011743                   

In [11]:

# Load both terrain datasets

elevation_df = pd.read_csv(
    "mumbai_terrain_elevation.csv"
)

geographical_df = pd.read_csv(
    "mumbai_terrain_geographical_features.csv"
)

# Rename waterway distance column
geographical_df = geographical_df.rename(
    columns={
        "distance_to_nullah_m":
        "distance_to_nearest_waterway_m"
    }
)

# Merge using zone_id
terrain_final_df = pd.merge(
    elevation_df,
    geographical_df,
    on=["zone_id", "zone_name"],
    how="inner"
)

# Validate
print(terrain_final_df)

print("\nMissing values:")
print(terrain_final_df.isnull().sum())

# Save final terrain dataset
terrain_final_df.to_csv(
    "mumbai_terrain_features.csv",
    index=False
)

print("\nTerrain dataset saved successfully!")


   zone_id   zone_name  latitude  longitude  elevation_m  distance_to_coast_m  \
0        0      Bandra     19.06      72.85          5.0          1777.105214   
1        1   Santacruz     19.08      72.84         12.0          1655.487095   
2        2  Vile Parle     19.10      72.85         19.0          2530.585895   
3        3     Andheri     19.12      72.85         19.0          3021.011743   
4        4  Jogeshwari     19.14      72.85         12.0          3876.400340   
5        5    Goregaon     19.16      72.85         12.0          5075.700192   
6        6       Malad     19.18      72.85         19.0          5669.582388   
7        7   Kandivali     19.20      72.85         20.0          5614.005591   
8        8    Borivali     19.23      72.85         19.0          6693.266465   

   distance_to_nearest_waterway_m  
0                      669.185896  
1                      652.891486  
2                      647.285367  
3                      755.981633  
4        